# Lazy image read example

Start `examples/python/fileTransfer/testDevice.py` in another process, then run this notebook. It connects to the fileTransfer `FileTransferDevice`, reads the image channels, and explicitly pulls the lazy BinaryData-backed image payload into the notebook.

In [1]:
from pathlib import Path

import stipy


config = stipy.Configuration({
    "Device Name": "FileTransferDevice Notebook Client",
    "IP Address": "localhost",
    "Module": "0",
    "Target Server": "sr-magis/2/Frame2",
})

config.set("NetworkHub", "NameService", "192.168.88.252:2809")
config.set("omniORB", "traceLevel", "0")
config.set("omniORB", "scanGranularity", "1")
config.set("omniORB", "clientConnectTimeOutPeriod", "500")
config.set("omniORB", "clientCallTimeOutPeriod", "2000")

device_id = stipy.DeviceID("FileTransferDevice", "localhost", 0, "sr-magis/2/Frame2")
device = stipy.connect(device_id, config=config)
device

++++ add( localhost/0/FileTransferDevice )


localhost/0/FileTransferDevice

++++ add( localhost/0/FileTransferDevice )


Channels 13, 14, and 15 return `Image`. Channel 13 is backed by inline `BinaryData`; channel 14 is backed by a raw `FileHolder`; channel 15 is backed by a TIF `FileHolder`. Remote reads return lightweight objects first, so the BinaryData-backed image can report its size before the bytes are pulled.

In [2]:
binary_image_read = device.read(13)
raw_file_image_read = device.read(14)
tif_file_image_read = device.read(15)

binary_image_read, raw_file_image_read, tif_file_image_read

(<Image | 10x10 | 100 bytes>, <Image | 10x10>, <Image | 10x10>)

The direct read result is lazy. The remote channel cache also stores the read result as `lastMeasurement`, using the same stream-backed representation for heavy BinaryData/Image channel state.

In [3]:
def last_image(channel_number):
    channel = device.getChannelManager().getChannel(channel_number)
    measurement = channel.getLastMeasurement()
    assert measurement.getType() == stipy.MixedValueType.Image
    return measurement.getImage()


direct_binary_image = binary_image_read
direct_raw_file_reference_image = raw_file_image_read
direct_tif_file_reference_image = tif_file_image_read
cached_binary_image = last_image(13)
cached_raw_file_reference_image = last_image(14)
cached_tif_file_reference_image = last_image(15)

direct_binary_image, direct_raw_file_reference_image, direct_tif_file_reference_image, cached_binary_image, cached_raw_file_reference_image, cached_tif_file_reference_image

(<Image | 10x10 | 100 bytes>,
 <Image | 10x10>,
 <Image | 10x10>,
 <Image | 10x10 | 100 bytes>,
 <Image | 10x10>,
 <Image | 10x10>)

In [4]:
def image_summary(image):
    data = image.getData() if image.hasData() else None
    file_id = image.getFileID()
    summary = {
        "width": image.getWidth(),
        "height": image.getHeight(),
        "hasData": image.hasData(),
        "hasFile": image.hasFile(),
        "fileID": {
            "origin": file_id.origin,
            "path": file_id.path,
            "filename": file_id.filename,
            "persistenceLocation": file_id.persistenceLocation,
        },
    }
    if data is not None:
        summary["data"] = {
            "bytes": data.bytes(),
            "length": data.length(),
            "wordsize": data.wordsize(),
            "hasStream": data.hasStream(),
            "hasLocalData": data.hasLocalData(),
            "isMaterialized": data.isMaterialized(),
        }
    return summary


{
    "directBinaryImage": image_summary(direct_binary_image),
    "directRawFileImage": image_summary(direct_raw_file_reference_image),
    "directTifFileImage": image_summary(direct_tif_file_reference_image),
    "cachedBinaryImage": image_summary(cached_binary_image),
    "cachedRawFileImage": image_summary(cached_raw_file_reference_image),
    "cachedTifFileImage": image_summary(cached_tif_file_reference_image),
}

{'directBinaryImage': {'width': 10,
  'height': 10,
  'hasData': True,
  'hasFile': False,
  'fileID': {'origin': '',
   'path': '',
   'filename': '',
   'persistenceLocation': ''},
  'data': {'bytes': 100,
   'length': 100,
   'wordsize': 1,
   'hasStream': True,
   'hasLocalData': False,
   'isMaterialized': False}},
 'directRawFileImage': {'width': 10,
  'height': 10,
  'hasData': False,
  'hasFile': False,
  'fileID': {'origin': 'localhost/0/FileTransferDevice',
   'path': 'localhost/0/FileTransferDevice/',
   'filename': 'fileTransfer-random-image-10.raw',
   'persistenceLocation': 'localhost/0/FileTransferDevice'}},
 'directTifFileImage': {'width': 10,
  'height': 10,
  'hasData': False,
  'hasFile': False,
  'fileID': {'origin': 'localhost/0/FileTransferDevice',
   'path': 'localhost/0/FileTransferDevice/',
   'filename': 'fileTransfer-random-image-11.tif',
   'persistenceLocation': 'localhost/0/FileTransferDevice'}},
 'cachedBinaryImage': {'width': 10,
  'height': 10,
  'hasDa

Explicitly pull the direct BinaryData-backed image read. Before `pull()`, the object reports stream metadata without local data. After `pull()`, the bytes are available in the notebook.

In [5]:
binary_data = direct_binary_image.getData()

before = {
    "bytes": binary_data.bytes(),
    "wordsize": binary_data.wordsize(),
    "hasStream": binary_data.hasStream(),
    "hasLocalData": binary_data.hasLocalData(),
    "isMaterialized": binary_data.isMaterialized(),
}

pulled = binary_data.pull()
payload = binary_data.getBytes()

after = {
    "pullSucceeded": pulled,
    "payloadLength": len(payload),
    "hasLocalData": binary_data.hasLocalData(),
    "isMaterialized": binary_data.isMaterialized(),
    "first16Bytes": list(payload[:16]),
}

before, after

({'bytes': 100,
  'wordsize': 1,
  'hasStream': True,
  'hasLocalData': False,
  'isMaterialized': False},
 {'pullSucceeded': True,
  'payloadLength': 100,
  'hasLocalData': True,
  'isMaterialized': True,
  'first16Bytes': [62,
   47,
   181,
   74,
   146,
   233,
   235,
   89,
   124,
   119,
   22,
   113,
   235,
   139,
   187,
   17]})

In [6]:
output_path = Path("pulled-fileTransfer-channel-13.raw")
output_path.write_bytes(payload)
output_path.resolve()

PosixPath('/home/hogan/code/dev/sti3/examples/python/fileTransfer/pulled-fileTransfer-channel-13.raw')

Optional display cell. The example image is raw 10x10 grayscale data, so this reshapes the pulled bytes into a small pixel array if NumPy and Matplotlib are available.

In [ ]:
try:
    import matplotlib.pyplot as plt
    import numpy as np

    pixels = np.frombuffer(payload, dtype=np.uint8).reshape(
        direct_binary_image.getHeight(),
        direct_binary_image.getWidth(),
    )
    plt.imshow(pixels, cmap="gray", vmin=0, vmax=255)
    plt.axis("off")
except ImportError:
    print("Install numpy and matplotlib to display the image inline.")

The FileHolder-backed images carry file identity metadata rather than inline lazy BinaryData. Channel 15 should report a `.tif` filename.

In [7]:
{
    "rawFileImage": image_summary(direct_raw_file_reference_image),
    "tifFileImage": image_summary(direct_tif_file_reference_image),
}

{'rawFileImage': {'width': 10,
  'height': 10,
  'hasData': False,
  'hasFile': False,
  'fileID': {'origin': 'localhost/0/FileTransferDevice',
   'path': 'localhost/0/FileTransferDevice/',
   'filename': 'fileTransfer-random-image-10.raw',
   'persistenceLocation': 'localhost/0/FileTransferDevice'}},
 'tifFileImage': {'width': 10,
  'height': 10,
  'hasData': False,
  'hasFile': False,
  'fileID': {'origin': 'localhost/0/FileTransferDevice',
   'path': 'localhost/0/FileTransferDevice/',
   'filename': 'fileTransfer-random-image-11.tif',
   'persistenceLocation': 'localhost/0/FileTransferDevice'}}}

Transfer the TIF FileHolder-backed image into a local virtual file holder. This keeps the materialized bytes in memory instead of writing them to disk.

In [ ]:
def transfer_image_to_virtual_holder(image):
    persistence = device.getPersistenceManager()
    file_server = persistence.getFileServer()
    virtual_file_server = persistence.makeVirtualFileServer()

    file_id = image.getFileID()
    assert file_server.findFile(file_id), "Source FileID is not available from the device file server. Restart examples/python/fileTransfer/testDevice.py so channels 14/15 use PersistenceManager.makeFileHolder()."
    backing_holder = stipy.VirtualFileHolder("notebook-virtual-file", file_id)
    destination_holder = persistence.makeVirtualFileHolder(backing_holder)

    assert file_server.transferFile(file_id, destination_holder, stipy.FileTransferType.Binary)

    virtual_file_server.addFile(backing_holder)
    virtual_file_id = backing_holder.getID()
    payload = backing_holder.getBytes()

    assert virtual_file_server.findFile(virtual_file_id)
    assert virtual_file_server.getFileSize(virtual_file_id) == len(payload)

    roundtrip_holder = stipy.VirtualFileHolder("notebook-virtual-roundtrip", virtual_file_id)
    assert virtual_file_server.transferFile(virtual_file_id, roundtrip_holder, stipy.FileTransferType.Binary)
    assert roundtrip_holder.getBytes() == payload

    return virtual_file_id, payload, roundtrip_holder.getBytes()


tif_file_id, tif_payload, tif_roundtrip_payload = transfer_image_to_virtual_holder(cached_tif_file_reference_image)

assert not cached_tif_file_reference_image.hasData()
assert cached_tif_file_reference_image.getFileID().filename.endswith(".tif")
assert tif_payload.startswith((b"II*\x00", b"MM\x00*"))

{
    "filename": tif_file_id.filename,
    "payloadLength": len(tif_payload),
    "startsWithTifMagic": tif_payload.startswith((b"II*\x00", b"MM\x00*")),
    "roundtripMatches": tif_roundtrip_payload == tif_payload,
}